# Guided Project: Schema Evolution & Data Validation

---

## 🌟 Overview: What is the purpose of this Lab?

Welcome to your advanced Data Engineering project! In modern data lakes, incoming data constantly changes. New columns are added, fields go missing, and bad data sneaks in. How do you prevent this from breaking your downstream systems?

**The Purpose of this Lab:**
You are part of a data engineering team responsible for processing evolving datasets. Your task is to process this data using **PySpark**, implement **Schema Evolution** to handle structural changes, apply **Data Validation** checks to ensure quality, and route invalid records into a **Quarantine (Dead Letter Queue)** location. Finally, you will automatically upload the cleaned and quarantined datasets to **Amazon S3**.

### Learning Path:
1.  **Authentication:** Configure your local environment to securely talk to AWS.
2.  **Data Engineering Foundations:** Master batch processing idempotency, schema evolution, and rule-based validation frameworks like Great Expectations.
3.  **Cloud Storage Basics:** Manually create an Amazon S3 Bucket to understand the destination.
4.  **The Validation Pipeline:** Build a step-by-step automated script to process evolving schemas, split clean/quarantine data, push them to the cloud, and log activities in CloudWatch.

---

## 📊 Dataset / Knowledge Source Used

Files: 
- `~/Desktop/Project/batch_v1.json` (Initial schema)
- `~/Desktop/Project/batch_v2.json` (Evolved schema with new columns)
- `~/Desktop/Project/transactions_raw.csv` (Historical transaction data)

Content:
- Financial transaction records containing structural drift (new fields over time) and data quality issues (negative amounts, missing IDs).

---

## 🛠️ Prerequisites & AWS Setup

Ensure the following are available:
- An **AWS account** with IAM permissions to use IAM, S3, and CloudWatch Logs.
- **VS Code** installed on your desktop (with Jupyter extensions pre-installed).
- The local data files provided in your workspace.

---

## Configure AWS Credentials

First, we need to log in to the AWS Web Console to set up our destination.

- Login to AWS Console:
  - Click on the `Lab Access` icon on the desktop.

![Images](images/lab-image1.png)

  - Click on `Access Lab` and using the given credentials login to AWS Console. This will allow you to access the AWS resources from the Console.

![Images](images/lab-image2.png)

  - Once you are logged in, set the region to `us-east-1`. All the resources to be created in **US-EAST-1 (N. Virginia) Region**

  - Steps to select AWS region in AWS Console.
    - Locate the region selector at the top-right corner of the console (next to your Account Name, which is painted in red).
    - Click on the dropdown, and a list of available AWS Regions will appear.

![Images](images/lab-image3.png)

    - Choose us-east-1 (N.Virginia) Region.

![Images](images/lab-image4.png)

---



## Validate & Configure Local CLI (The Bridge to AWS)

**Purpose of this Activity:** To ensure your local terminal is authenticated with AWS. We validate first, and only configure if validation fails.

### Step 1: Validate Existing AWS Configuration
Let's make sure your setup was successful before we proceed. We will use the `aws sts get-caller-identity` command to verify AWS knows who you are.


In [ ]:
# Press shift + ` to open terminal in VS code.
# 👇 Select and copy the command below, then paste it into your VS Code terminal 👇
# aws sts get-caller-identity

If configured correctly, you will see a JSON response in your terminal containing your `UserId`, `Account`, and `Arn`.

![Images](images/aws_configure_test.png)

### Step 2: Configure AWS (If Validation Fails)
If the command above threw an error, you must configure your credentials.
Locate the side panel on your lab platform. Click on the **Lab Credentials** tab or expand the **Cloudx Account Info** section by clicking the arrow to reveal your **AccessKeyID** and **SecretAccessKey**.

![Images](images/aws_configure1.png)

![Images](images/aws_configure2.png)


In [ ]:
# Press shift + ` to open the terminal in VS code.
# 👇 Select and copy the command below, then paste it into your VS Code terminal 👇
# aws configure

When prompted in the terminal, enter your credentials exactly like this:
```text
AWS Access Key ID [None]: <Paste your AccessKeyID here>
AWS Secret Access Key [None]: <Paste your SecretAccessKey here>
Default region name [None]: us-east-1
Default output format [None]: json
```

![Images](images/aws_configure.png)

---

# Activity 1: Foundations of PySpark & Data Validation

**Purpose of this Activity:** To ensure your big data processing fundamentals are production-ready. Data Engineers must understand batch processing, handling evolving schemas gracefully, and applying strict validation rules to maintain data lakes.



Now, we initialize our PySpark session.

💡 **What this code does:**
- Initializes a PySpark session, which is required to start working with big data using Spark.
- Sets up file paths for two JSON datasets (batch_v1.json and batch_v2.json) so they can be loaded and processed later.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os

# Initialize Spark Session (The entry point for PySpark)
spark = SparkSession.builder \
    .appName("SchemaEvolutionLab") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

# Suppress WARN-level logs (e.g., NativeCodeLoader warning)
spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Initialized!")

# Define file paths for the labs
v1_path = os.path.expanduser("~/Desktop/Project/batch_v1.json")
v2_path = os.path.expanduser("~/Desktop/Project/batch_v2.json")
raw_csv_path = os.path.expanduser("~/Desktop/Project/transactions_raw.csv")

![Images](images/pyspark_check.png)

### Pillar 1: Batch Processing & Idempotency
Batch jobs process massive amounts of data at scheduled intervals. Designing jobs to be *idempotent* means you can safely re-run them without duplicating data.

💡 **What this code does:**
- Reads our historical transaction CSV into a PySpark DataFrame.
- Drops exact duplicate rows (`dropDuplicates()`) to ensure re-running the job doesn't corrupt downstream tables.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Read batch data
df_raw = spark.read.csv(raw_csv_path, header=True, inferSchema=True)

# Ensure idempotency by dropping exact duplicates
df_idempotent = df_raw.dropDuplicates()

print(f"Original Count: {df_raw.count()} | Idempotent Count: {df_idempotent.count()}")
df_idempotent.show(5)

![Images](images/pillar1_result.png)

> 💡 **TODO: Guided Practice**
> Complete the code below to write a function that takes a file path, loads a JSON file using PySpark, and returns the total row count.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# TODO: Create a function to load a JSON file and count its records
def count_json_records(file_path):
    result = None
    
    # code starts here

    # TODO: Read the JSON file at 'file_path' using spark.read.json()
    # TODO: Calculate the total rows using .count() and assign it to 'result'
    
    # code ends here
    
    return result

total_records = count_json_records(v1_path)
# TODO: Uncomment the line below to print the total records in batch 1
# print("Total Records in Batch 1:\n", total_records)

> 🎯 **Try Out: Independent Challenge**
> Write a PySpark snippet entirely from scratch below. Given `df_idempotent`, use the `.filter()` method to extract only the rows where the `user_id` is exactly `'U881'`. Show the results using `.show()`.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!




### Pillar 2: Schema Evolution
Data schemas change over time. `batch_v2.json` contains new columns (`email`, `category`, `status`, etc.) that are not in `batch_v1.json`. Standard readers crash if schemas mismatch. PySpark's `mergeSchema` option handles schema drift seamlessly.

💡 **What this code does:**
- Reads a list of files with different structures.
- Instructs Spark to dynamically merge their schemas using `.option("mergeSchema", "true")`.
- Automatically injects `null` values for older records that lack the newly evolved columns.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Merge evolving schemas safely
evolved_df = spark.read.option("mergeSchema", "true").json([v1_path, v2_path])

print("--- Merged Evolved Schema ---")
evolved_df.printSchema()

print("\n--- Note the nulls injected for older records lacking new fields ---")
evolved_df.select("transaction_id", "amount", "email", "category").show(5)

![Images](images/pillar2_one.png)
![Images](images/pillar2_two.png)

> 💡 **TODO: Guided Practice**
> Complete the code below to write a reusable function that reads any list of files with schema merging enabled.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# TODO: Implement a function to read merged schemas
def read_with_schema_evolution(file_paths_list):
    result = None
    
    # code starts here

    # TODO: Use spark.read.option("mergeSchema", "true").json() passing the file_paths_list
    # TODO: Store the output dataframe in 'result'

    # code ends here
    
    return result

merged_data = read_with_schema_evolution([v1_path, v2_path])
# TODO: Uncomment the line below to print the total evolved records
# print("Total Evolved Records:\n", merged_data.count())

> 🎯 **Try Out: Independent Challenge**
> Because we merged schemas, many older records have `null` in the new `'email'` column. Write a PySpark snippet below to fill all `null` values in the `evolved_df` specifically in the `'email'` column with the string `'UNKNOWN'`. Use `.fillna()` and `.show()` the results.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!




### Pillar 3: Data Validation Frameworks
Data quality is paramount. Frameworks like **Great Expectations (GE)** and native PySpark rules ensure bad data is caught early before it poisons your data lake.

💡 **What this code does:**
- Imports `great_expectations`.
- Converts a small sample of our data into a GE Pandas dataset for demonstration.
- Applies a strict rule (`expect_column_values_to_be_between`) to ensure `amount >= 0`.
- Prints a detailed validation report that stakeholders can read.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
import great_expectations as ge
from great_expectations.dataset import PandasDataset

sample_pdf = evolved_df.limit(100).toPandas()

ge_df = PandasDataset(sample_pdf)

result = ge_df.expect_column_values_to_be_between(
    column="amount",
    min_value=0
)

print(result)

![Images](images/pillar3_one.png)
![Images](images/pillar3_two.png)

> 💡 **TODO: Guided Practice**
> While GE provides great reporting, PySpark's `.filter()` is used heavily for native, large-scale data routing (Quarantine Pattern). Complete the code below to implement a validation rule that filters out records where the `user_id` is missing (`isNull()`).
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# TODO: Implement a function to find records missing a user_id
def find_missing_users(dataframe):
    result = None
    
    # code starts here

    # TODO: Use dataframe.filter() and col("user_id").isNull() to find bad records
    # TODO: Assign the filtered dataframe to 'result'

    # code ends here
    
    return result

missing_user_df = find_missing_users(evolved_df)
# TODO: Uncomment the line below to print the count of records with missing user IDs
# print("Records with Missing User IDs:\n", missing_user_df.count())

> 🎯 **Try Out: Independent Challenge**
> Write a snippet from scratch to extract only the records from `evolved_df` where the `status` is exactly equal to `'failed'`. Since data can be messy, use the `lower()` function from `pyspark.sql.functions` to handle variations like 'FAILED' or 'Failed'.
>
> **⚡ How to run the code:** Write your solution in the blank cell, select it, and press **Shift + Enter**.

In [ ]:
# Write your Try Out solution here!




---

# Activity 2: Create Amazon S3 Bucket via AWS Console

**Purpose of this Activity:** To manually create the target storage infrastructure in the cloud so you understand where your automated pipeline will send its processed (Clean vs Quarantined) data.

1.  In the AWS Management Console search bar, type **S3** and select it from the services menu.

![Images](images/1search_click_s3.png)

2.  Click the **Create bucket** button.

![Images](images/2create_bucket.png)

3.  **Bucket Name Configuration:**
    - Provide a globally unique name. To easily format and copy your bucket name, we have provided a helper script below.

💡 **What this code does:**
- Stores your unique identifier in a variable.
- Generates a properly formatted AWS S3 Bucket name.
- Prints the name so you can easily copy it for the AWS Console.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# Edit the string below with your unique identifier (e.g., your name and a number)
my_unique_id = "<REPLACE WITH YOUR UNIQUE ID>" # <-- EDIT THIS

# We use f-strings in Python to combine text and variables
my_bucket_name = f"s3-data-quarantine-{my_unique_id}"

print("👇 Select and copy the text below to use as your Bucket Name 👇")
print(my_bucket_name)

![Images](images/copy_bucket_name.png)

4.  **AWS Region:** Ensure it is set to **us-east-1 (N. Virginia)**.
5.  **Block Public Access settings for this bucket:** - Uncheck the **Block all public access** box.
    - Check the warning box below it that says "I acknowledge that the current settings might result in this bucket and the objects within becoming public."
6.  Scroll to the bottom and click **Create bucket**.

![Images](images/3bucket_name_region.jpeg)

---

# Activity 3: The Validation & Quarantine Pipeline (Step-by-Step)

**Purpose of this Activity:** To build a complete, professional-grade pipeline that implements the "Dead Letter Queue" or Quarantine pattern. It reads evolving pipeline metrics, applies quality rules, splits the data, uploads the results to S3, and logs everything to Amazon CloudWatch.

⚠️ **Run the steps in order**. Each cell depends on functions and variables defined in the one before it.

### Step 3.1: Set up CloudWatch Logging Functions
A professional script shouldn't just `print()` results; it should log them to the cloud. We will define functions to push logs to Amazon CloudWatch.

💡 **What this code does:**
- Connects to the AWS `logs` service using Boto3.
- Checks if a specific log group (`/aws/custom/validation-logs`) exists, and creates it if not.
- Defines a reusable function `send_log_to_cloudwatch` that timestamps our messages and pushes them securely to AWS.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
import time
import boto3
from botocore.exceptions import ClientError

logs_client = boto3.client('logs', region_name='us-east-1')

LOG_GROUP = "/aws/custom/validation-logs"
LOG_STREAM = "ExecutionStream"

def setup_cloudwatch_logs():
    """Create Log Group and Stream in CloudWatch if they don't exist."""
    try:
        logs_client.create_log_group(logGroupName=LOG_GROUP)
    except ClientError: pass # Ignore if it already exists

    try:
        logs_client.create_log_stream(logGroupName=LOG_GROUP, logStreamName=LOG_STREAM)
    except ClientError: pass

def send_log_to_cloudwatch(message):
    """Print locally and send log message to AWS CloudWatch Logs."""
    print(f"INFO: {message}")
    timestamp = int(round(time.time() * 1000))
    try:
        logs_client.put_log_events(
            logGroupName=LOG_GROUP,
            logStreamName=LOG_STREAM,
            logEvents=[{'timestamp': timestamp, 'message': message}]
        )
    except Exception as e:
        print(f"Failed to send log to CloudWatch: {e}")

setup_cloudwatch_logs()
send_log_to_cloudwatch("CloudWatch setup complete. Ready for pipeline execution.")

![Images](images/activity3_1.png)

### Step 3.2: Load Evolving Data
We will load both batch JSONs, enabling schema evolution.

💡 **What this code does:**
- Reads `batch_v1.json` and `batch_v2.json` simultaneously with `mergeSchema=True`.
- Logs the successful read action to CloudWatch.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
def load_pipeline_data():
    try:
        df = spark.read.option("mergeSchema", "true").json([v1_path, v2_path])
        send_log_to_cloudwatch(f"Successfully merged evolving schemas. Total rows: {df.count()}")
        return df
    except Exception as e:
        send_log_to_cloudwatch(f"Error loading data: {e}")
        return None

master_df = load_pipeline_data()

![Images](images/activity3_2.png)

### Step 3.3: Implement the Quarantine Pattern (TODO)
We need to separate healthy data from corrupted data.

> 💡 **TODO: Guided Practice**
> Complete the code below to split `master_df` into two dataframes. `clean_df` should contain rows where `amount >= 0`. `quarantine_df` should contain rows where `amount < 0`.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# TODO: Create a function to split clean data from quarantined data
def split_clean_and_quarantine(dataframe):
    result = None
    
    # code starts here
    
    # TODO: Filter for rows where 'amount' >= 0, assign to clean_data
    # TODO: Filter for rows where 'amount' < 0, assign to quarantine_data
    # TODO: Assign a tuple of (clean_data, quarantine_data) to 'result'
    
    # code ends here
    
    return result

clean_df, quarantine_df = split_clean_and_quarantine(master_df)

# TODO: Uncomment the lines below to log the split results to CloudWatch and print locally
# send_log_to_cloudwatch(f"Data split complete. Clean: {clean_df.count()}, Quarantined: {quarantine_df.count()}")
# print(f"Split complete. Clean records: {clean_df.count()}, Quarantined: {quarantine_df.count()}")

### Step 3.4: Format S3 Target Zones (TODO)
Before uploading, we need to generate different S3 folder paths (prefixes) based on data quality.

> 💡 **TODO: Guided Practice**
> Complete the code below to format the S3 upload key dynamically.
>
> **NOTE:** Please ensure you complete and execute the code cell below without changing the function name, as your results will be validated and automatically reflected in your Practice Progress Summary to track your learning milestones.
>
> **⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
# TODO: Create a function to format the S3 key zone
def get_s3_zone_key(zone_name, file_name):
    result = None
    
    # code starts here
    
    # TODO: Combine zone_name and file_name using an f-string (e.g., "zone_name/file_name")
    # TODO: Assign it to 'result'
    
    # code ends here
    
    return result

clean_key = get_s3_zone_key('clean_zone', 'data.parquet')

# TODO: Uncomment the lines below to generate the quarantine key and print both keys
# quarantine_key = get_s3_zone_key('quarantine_zone', 'data.parquet')
# print("Upload Destinations:\n", clean_key, "\n", quarantine_key)

### Step 3.5: Upload Processed Data to S3
Now we push our categorized data to the cloud bucket using Pandas and Boto3. 

💡 **What this code does:**
- Converts PySpark dataframes to Pandas (to write them easily as single `.parquet` files).
- Pushes `clean_data.parquet` and `quarantine_data.parquet` to their respective zones in S3.
- Logs the success or failure to CloudWatch.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
import boto3
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

file_v1 = os.path.expanduser("~/Desktop/Project/batch_v1.json")
file_v2 = os.path.expanduser("~/Desktop/Project/batch_v2.json")

df_v1 = spark.read.json(file_v1)
df_v2 = spark.read.json(file_v2)

df_merged = df_v1.unionByName(df_v2, allowMissingColumns=True)

TARGET_BUCKET = my_bucket_name

clean_key = "clean_zone/clean_data.parquet"
quarantine_key = "quarantine_zone/quarantine_data.parquet"

valid_condition = col("user_id").isNotNull() & (col("amount") > 0)

clean_df = df_merged.filter(valid_condition)
quarantine_df = df_merged.filter(~valid_condition)

s3_client = boto3.client('s3', region_name='us-east-1')

def save_and_upload(spark_df, local_name, target_key):
    try:
        spark_df.toPandas().to_parquet(local_name)
        s3_client.upload_file(local_name, TARGET_BUCKET, target_key)
        print(f"Uploaded: {target_key}")
    except Exception as e:
        print(f"Upload failed: {e}")

save_and_upload(clean_df, 'clean_data.parquet', clean_key)
save_and_upload(quarantine_df, 'quarantine_data.parquet', quarantine_key)

![Images](images/activity3_5.png)

### Step 3.6: Verify the Upload via Boto3
Finally, we query S3 to list its contents and prove our routing arrived safely.

💡 **What this code does:**
- Uses `s3_client.list_objects_v2` to ask AWS what is currently inside our bucket.
- Iterates through the response and logs the name and size of each file it finds.

**⚡ How to run the code:** Select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code.

In [ ]:
def list_s3_objects(target_bucket):
    """Lists objects in the S3 bucket to verify the upload."""
    try:
        response = s3_client.list_objects_v2(Bucket=target_bucket)
        if 'Contents' in response:
            for obj in response['Contents']:
                send_log_to_cloudwatch(f"Found object in S3: {obj['Key']} (Size: {obj['Size']} bytes)")
        else:
            send_log_to_cloudwatch("No objects found in bucket.")
    except ClientError as e:
        send_log_to_cloudwatch(f"Listing objects failed: {e}")

list_s3_objects(TARGET_BUCKET)
send_log_to_cloudwatch("Validation Utility script execution completed successfully.")

![Images](images/activity3_6.png)

---

# Activity 4: Verify S3 Upload via AWS Console

**Purpose of this Activity:** To visually confirm that your automated script successfully placed the separated parquet files into the correct cloud zones.

1.  Return to the **AWS Management Console**.
2.  Navigate to the **S3** service.
3.  Click on your bucket.

![Images](images/navigate_s3.png)

4.  You should see folders named `clean_zone` and `quarantine_zone`.

![Images](images/folders.png)

5.  Click inside them to verify that the `.parquet` data files are present. This completes the console verification of your programmatic routing.

![Images](images/parquet1.png)
![Images](images/parquet2.png)

---

# Activity 5: Navigate to and Verify CloudWatch Logs

**Purpose of this Activity:** Professional, advanced-level integration requires non-optional, structured logging in the cloud for full auditability. In this activity, you MUST verify that your notebook's entire execution trail was captured in real-time in Amazon CloudWatch.

1.  In the AWS Management Console search bar, type **CloudWatch** and select it.

![Images](images/19search_cloudwatch.png)

2.  On the left-hand navigation pane, expand **Logs** and click on **Log Management**.

![Images](images/20select_log_management.png)

3.  Under **Log groups** You will see your logs created by your script: `/aws/custom/validation-logs`. This unique log group is a professional standard practice for automated scripts.

![Images](images/validation_logs.png)

4.  Click on the log group name.
5.  Scroll down, Under the **Log streams** tab, click on the stream named `ExecutionStream`.

![Images](images/stream.png)

6.  You MUST review the log events. Verify that you see a complete sequential trail of your Jupyter Notebook execution. The wordings may differ, so the logs may be exactly the same as below or have a similar meaning whatever is shown in the console:
    - _Successfully merged evolving schemas..._
    - _Data split complete. Clean: X, Quarantined: Y_
    - _File clean_data.parquet uploaded to S3 zone..._
    - _Found object in S3..._

![Images](images/logs.png)

---

### 🧠 Try Out: Recover Quarantined Data
In the real world, Data Engineers often have to fix quarantined data and send it back into the pipeline. 
Write a quick script below to:
1. Take the `quarantine_df` we generated earlier.
2. Use PySpark to fill any `null` values in the `user_id` column with the string `'RECOVERED_USER'`.
3. Save the result locally as `recovered_data.parquet` (using `.toPandas().to_parquet(...)`).
4. Upload it to your S3 bucket under the folder `recovered_zone/recovered_data.parquet`.

**⚡ How to run the code:** Write your solution, select the below code cell and press **Shift + Enter** or click the **Play (▶)** button on the top left side of the code block to run the code. *(Solution provided at the bottom of the notebook!)*

In [ ]:
# Write your Try Out solution here!




---

## **Please ensure that the file is saved before exiting, as this is required for validation**

## 🎓 Conclusion

This comprehensive guided project demonstrated a robust, real-world data engineering workflow. You started by applying fundamental PySpark skills to process batch data, handle micro-batches, and manage Upserts/CDC workflows.

Most importantly, you handled **Schema Evolution** gracefully using `mergeSchema` and implemented a professional **Data Validation / Quarantine** pipeline. By successfully closing the loop and automating the routing of Clean vs. Corrupted data into **Amazon S3** using **Boto3**, and logging the results via **Amazon CloudWatch**, you have demonstrated the exact skills required to maintain reliable, enterprise-grade data lakes.

---
